Set up

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import numpy as np
import time
import matplotlib.pyplot as plt
from torchvision import datasets, transforms
from tqdm import tqdm

use_cuda = torch.cuda.is_available() 
device = torch.device("cuda" if use_cuda else "cpu")
batch_size = 64

np.random.seed(42)
torch.manual_seed(42)

# print(f"Using device: {device}")

## Dataloaders - No normalization for IBP
train_dataset = datasets.MNIST('mnist_data/', train=True, download=True, 
                              transform=transforms.Compose([transforms.ToTensor()]))
test_dataset = datasets.MNIST('mnist_data/', train=False, download=True, 
                             transform=transforms.Compose([transforms.ToTensor()]))

train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

## Correct architecture as per Problem 2 requirements
class IBPNet(nn.Module):
    def __init__(self):
        super(IBPNet, self).__init__()
        # 3 layers, each with 50 neurons as specified
        self.fc1 = nn.Linear(28*28, 50)
        self.fc2 = nn.Linear(50, 50)
        self.fc3 = nn.Linear(50, 50)
        self.fc4 = nn.Linear(50, 10)  # Output layer
        
    def forward(self, x):
        x = x.view((-1, 28*28))
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = F.relu(self.fc3(x))
        x = self.fc4(x)  # No softmax! Return logits
        return x

Using device: cuda


In [4]:
model = IBPNet().to(device)
model.train()

IBPNet(
  (fc1): Linear(in_features=784, out_features=50, bias=True)
  (fc2): Linear(in_features=50, out_features=50, bias=True)
  (fc3): Linear(in_features=50, out_features=50, bias=True)
  (fc4): Linear(in_features=50, out_features=10, bias=True)
)

In [5]:
# Use the model already created in the previous cell
print("Model architecture:")
print(model)
print(f"\nTotal parameters: {sum(p.numel() for p in model.parameters())}")

# Verify the architecture
for name, layer in model.named_modules():
    if isinstance(layer, nn.Linear):
        print(f"{name}: {layer}")

Model architecture:
IBPNet(
  (fc1): Linear(in_features=784, out_features=50, bias=True)
  (fc2): Linear(in_features=50, out_features=50, bias=True)
  (fc3): Linear(in_features=50, out_features=50, bias=True)
  (fc4): Linear(in_features=50, out_features=10, bias=True)
)

Total parameters: 44860
fc1: Linear(in_features=784, out_features=50, bias=True)
fc2: Linear(in_features=50, out_features=50, bias=True)
fc3: Linear(in_features=50, out_features=50, bias=True)
fc4: Linear(in_features=50, out_features=10, bias=True)


In [ ]:
# Install if needed (uncomment if necessary)
# !pip install bound_propagation

from bound_propagation import BoundModelFactory, HyperRectangle

# Verify the factory works
factory = BoundModelFactory()

bound_propagation imported successfully!
BoundModelFactory created successfully!


equation 12

In [10]:
def compute_worst_case_logits(model, x, y, epsilon):
    """
    Compute worst-case logits using bound_propagation library
    Returns the cross-entropy loss on worst-case logits
    """
    batch_size = x.shape[0]
    x_flat = x.view(batch_size, -1)
    
    # Create input bounds: [x - epsilon, x + epsilon] clipped to [0, 1]
    x_l = torch.clamp(x_flat - epsilon, min=0, max=1)
    x_u = torch.clamp(x_flat + epsilon, min=0, max=1)
    
    # Create hyperrectangle for IBP
    input_bounds = HyperRectangle(x_l, x_u)
    
    # Create a sequential model for bound_propagation
    # We need to manually construct the layers
    bound_model = nn.Sequential(
        model.fc1,
        nn.ReLU(),
        model.fc2,
        nn.ReLU(),
        model.fc3,
        nn.ReLU(),
        model.fc4
    )
    
    # Build bounded model
    factory = BoundModelFactory()
    bounded_net = factory.build(bound_model)
    
    # Perform IBP to get logit bounds
    logit_bounds = bounded_net.ibp(input_bounds)
    
    # Extract lower and upper bounds
    logit_l = logit_bounds.lower
    logit_u = logit_bounds.upper
    
    # Construct worst-case logits (Equation 9 from paper)
    # For true class: use lower bound (minimize it)
    # For other classes: use upper bound (maximize them)
    worst_case_logits = torch.zeros_like(logit_l)
    for i in range(batch_size):
        true_class = y[i]
        for c in range(logit_l.shape[1]):
            if c == true_class:
                worst_case_logits[i, c] = logit_l[i, c]
            else:
                worst_case_logits[i, c] = logit_u[i, c]
    
    return F.cross_entropy(worst_case_logits, y)

# Test the function with a small batch
test_x, test_y = next(iter(train_loader))
test_x, test_y = test_x[:4].to(device), test_y[:4].to(device)

print("Testing worst-case logits computation...")
robust_loss = compute_worst_case_logits(model, test_x, test_y, epsilon=0.05)
print(f"Robust loss computed: {robust_loss.item():.4f}")
print("Function works correctly!")

Testing worst-case logits computation...
Robust loss computed: 6.6699
Function works correctly!


Part a

In [ ]:
#Part a, ibp training
def train_ibp(model, train_loader, num_epochs=20, target_epsilon=0.1):
    """
    Train with IBP using gradual epsilon and kappa scheduling
    Following the paper's training tricks
    """
    optimizer = optim.Adam(model.parameters(), lr=0.001)
    
    training_history = {
        'loss': [],
        'standard_loss': [],
        'robust_loss': [],
        'kappa': [],
        'epsilon': []
    }
    
    print(f"Starting IBP training for {num_epochs} epochs...")
    print(f"Target epsilon: {target_epsilon}")
    print(f"Kappa schedule: 1.0 -> 0.5")
    print("-" * 70)
    
    for epoch in range(num_epochs):
        model.train()
        epoch_loss = 0
        epoch_std_loss = 0
        epoch_rob_loss = 0
        
        # Schedule kappa: linearly decrease from 1.0 to 0.5
        kappa = 1.0 - 0.5 * (epoch / num_epochs)
        
        # Schedule epsilon: linearly increase from 0 to target_epsilon
        epsilon = target_epsilon * (epoch / num_epochs)
        
        num_batches = 0
        for batch_idx, (x, y) in enumerate(train_loader):
            x, y = x.to(device), y.to(device)
            
            optimizer.zero_grad()
            
            # Standard loss term: κ * CE(z^K, y_true)
            logits = model(x)
            standard_loss = F.cross_entropy(logits, y)
            
            # Robust loss term: (1-κ) * CE(z_hat^K(epsilon), y_true)
            robust_loss = compute_worst_case_logits(model, x, y, epsilon)
            
            # Combined loss (Equation 12 from paper)
            loss = kappa * standard_loss + (1 - kappa) * robust_loss
            
            loss.backward()
            optimizer.step()
            
            epoch_loss += loss.item()
            epoch_std_loss += standard_loss.item()
            epoch_rob_loss += robust_loss.item()
            num_batches += 1
            
            # Print progress every 100 batches
            if (batch_idx + 1) % 100 == 0:
                print(f"Epoch {epoch+1}/{num_epochs}, Batch {batch_idx+1}/{len(train_loader)}, "
                      f"Loss: {loss.item():.4f}, κ: {kappa:.3f}, ε: {epsilon:.4f}")
        
        # Record epoch statistics
        avg_loss = epoch_loss / num_batches
        avg_std_loss = epoch_std_loss / num_batches
        avg_rob_loss = epoch_rob_loss / num_batches
        
        training_history['loss'].append(avg_loss)
        training_history['standard_loss'].append(avg_std_loss)
        training_history['robust_loss'].append(avg_rob_loss)
        training_history['kappa'].append(kappa)
        training_history['epsilon'].append(epsilon)
        
        print(f"Epoch {epoch+1}/{num_epochs} Summary:")
        print(f"  Avg Loss: {avg_loss:.4f} | Std Loss: {avg_std_loss:.4f} | "
              f"Rob Loss: {avg_rob_loss:.4f} | κ: {kappa:.3f} | ε: {epsilon:.4f}")
        print("-" * 70)
    
    return training_history

# Train the IBP model
print("="*70)
print("TRAINING IBP MODEL")
print("="*70)
start_time = time.time()
history = train_ibp(model, train_loader, num_epochs=20, target_epsilon=0.1)
ibp_training_time = time.time() - start_time

print(f"\n{'='*70}")
print(f"IBP Training completed in {ibp_training_time:.2f} seconds ({ibp_training_time/60:.2f} minutes)")
print(f"{'='*70}")

TRAINING IBP MODEL
Starting IBP training for 20 epochs...
Target epsilon: 0.1
Kappa schedule: 1.0 -> 0.5
----------------------------------------------------------------------
Epoch 1/20, Batch 100/938, Loss: 0.8263, κ: 1.000, ε: 0.0000
Epoch 1/20, Batch 200/938, Loss: 0.3649, κ: 1.000, ε: 0.0000
Epoch 1/20, Batch 300/938, Loss: 0.1973, κ: 1.000, ε: 0.0000
Epoch 1/20, Batch 400/938, Loss: 0.1264, κ: 1.000, ε: 0.0000
Epoch 1/20, Batch 500/938, Loss: 0.2152, κ: 1.000, ε: 0.0000
Epoch 1/20, Batch 600/938, Loss: 0.3184, κ: 1.000, ε: 0.0000
Epoch 1/20, Batch 700/938, Loss: 0.4298, κ: 1.000, ε: 0.0000
Epoch 1/20, Batch 800/938, Loss: 0.1584, κ: 1.000, ε: 0.0000
Epoch 1/20, Batch 900/938, Loss: 0.2447, κ: 1.000, ε: 0.0000
Epoch 1/20 Summary:
  Avg Loss: 0.4326 | Std Loss: 0.4326 | Rob Loss: 0.4326 | κ: 1.000 | ε: 0.0000
----------------------------------------------------------------------
Epoch 2/20, Batch 100/938, Loss: 0.3026, κ: 0.975, ε: 0.0050
Epoch 2/20, Batch 200/938, Loss: 0.2257, κ:

In [ ]:
# part a, train a standard model for comparison
print("="*70)
print("TRAINING STANDARD BASELINE MODEL")
print("="*70)

standard_model = IBPNet().to(device)
optimizer = optim.Adam(standard_model.parameters(), lr=0.001)

start_time = time.time()

for epoch in range(20):
    standard_model.train()
    epoch_loss = 0
    num_batches = 0
    
    for batch_idx, (x, y) in enumerate(train_loader):
        x, y = x.to(device), y.to(device)
        
        optimizer.zero_grad()
        logits = standard_model(x)
        loss = F.cross_entropy(logits, y)
        loss.backward()
        optimizer.step()
        
        epoch_loss += loss.item()
        num_batches += 1
        
        if (batch_idx + 1) % 100 == 0:
            print(f"Epoch {epoch+1}/20, Batch {batch_idx+1}/{len(train_loader)}, Loss: {loss.item():.4f}")
    
    avg_loss = epoch_loss / num_batches
    print(f"Epoch {epoch+1}/20 Summary: Avg Loss: {avg_loss:.4f}")
    print("-" * 70)

standard_training_time = time.time() - start_time

print(f"\n{'='*70}")
print(f"Standard training completed in {standard_training_time:.2f} seconds ({standard_training_time/60:.2f} minutes)")
print(f"{'='*70}")

print(f"\n{'='*70}")
print("TRAINING TIME COMPARISON")
print(f"{'='*70}")
print(f"IBP Training Time:      {ibp_training_time:.2f} seconds ({ibp_training_time/60:.2f} minutes)")
print(f"Standard Training Time: {standard_training_time:.2f} seconds ({standard_training_time/60:.2f} minutes)")
print(f"Difference:             {ibp_training_time - standard_training_time:.2f} seconds ({(ibp_training_time - standard_training_time)/60:.2f} minutes)")
print(f"IBP is {ibp_training_time/standard_training_time:.2f}x slower than standard training")
print(f"{'='*70}")

TRAINING STANDARD BASELINE MODEL
Epoch 1/20, Batch 100/938, Loss: 0.5520
Epoch 1/20, Batch 200/938, Loss: 0.5900
Epoch 1/20, Batch 300/938, Loss: 0.1880
Epoch 1/20, Batch 400/938, Loss: 0.2318
Epoch 1/20, Batch 500/938, Loss: 0.3264
Epoch 1/20, Batch 600/938, Loss: 0.2402
Epoch 1/20, Batch 700/938, Loss: 0.1848
Epoch 1/20, Batch 800/938, Loss: 0.1922
Epoch 1/20, Batch 900/938, Loss: 0.1657
Epoch 1/20 Summary: Avg Loss: 0.4436
----------------------------------------------------------------------
Epoch 2/20, Batch 100/938, Loss: 0.3250
Epoch 2/20, Batch 200/938, Loss: 0.2004
Epoch 2/20, Batch 300/938, Loss: 0.1200
Epoch 2/20, Batch 400/938, Loss: 0.2444
Epoch 2/20, Batch 500/938, Loss: 0.2943
Epoch 2/20, Batch 600/938, Loss: 0.2400
Epoch 2/20, Batch 700/938, Loss: 0.2137
Epoch 2/20, Batch 800/938, Loss: 0.0710
Epoch 2/20, Batch 900/938, Loss: 0.2206
Epoch 2/20 Summary: Avg Loss: 0.1821
----------------------------------------------------------------------
Epoch 3/20, Batch 100/938, Loss

In [ ]:
#part a, attack on both models
def pgd_attack(model, x, y, epsilon=0.1, alpha=0.01, num_iter=40):
    """
    Perform PGD attack (multi-step FGSM)
    """
    model.eval()
    x_flat = x.view(x.shape[0], -1)
    x_adv_flat = x_flat.clone().detach()
    
    for _ in range(num_iter):
        x_adv_flat.requires_grad = True
        
        # Reconstruct image shape for forward pass
        x_adv = x_adv_flat.view_as(x)
        logits = model(x_adv)
        loss = F.cross_entropy(logits, y)
        
        # Compute gradient
        grad = torch.autograd.grad(loss, x_adv_flat)[0]
        
        # Update adversarial example
        x_adv_flat = x_adv_flat.detach() + alpha * grad.sign()
        
        # Project back to epsilon ball and [0,1]
        perturbation = torch.clamp(x_adv_flat - x_flat, min=-epsilon, max=epsilon)
        x_adv_flat = torch.clamp(x_flat + perturbation, min=0, max=1)
    
    return x_adv_flat.view_as(x)

def evaluate_accuracy_and_robustness(model, test_loader, epsilon=0.1, model_name="Model"):
    """
    Evaluate both standard and robust accuracy (PGD)
    """
    model.eval()
    correct_standard = 0
    correct_robust = 0
    total = 0
    
    print(f"\nEvaluating {model_name}...")
    for x, y in tqdm(test_loader, desc='Testing'):
        x, y = x.to(device), y.to(device)
        
        # Standard accuracy
        with torch.no_grad():
            logits = model(x)
            pred = logits.argmax(dim=1)
            correct_standard += (pred == y).sum().item()
        
        # Robust accuracy (PGD attack)
        x_adv = pgd_attack(model, x, y, epsilon=epsilon, alpha=0.01, num_iter=40)
        with torch.no_grad():
            logits_adv = model(x_adv)
            pred_adv = logits_adv.argmax(dim=1)
            correct_robust += (pred_adv == y).sum().item()
        
        total += y.size(0)
    
    standard_acc = 100 * correct_standard / total
    robust_acc = 100 * correct_robust / total
    
    return standard_acc, robust_acc

# Evaluate both models
print("="*70)
print("PART (a): EVALUATION - Standard and Robust Accuracy")
print("="*70)

ibp_std_acc, ibp_rob_acc = evaluate_accuracy_and_robustness(
    model, test_loader, epsilon=0.1, model_name="IBP Model"
)

std_std_acc, std_rob_acc = evaluate_accuracy_and_robustness(
    standard_model, test_loader, epsilon=0.1, model_name="Standard Model"
)

print("\n" + "="*70)
print("RESULTS SUMMARY - PART (a)")
print("="*70)
print(f"\nIBP Model:")
print(f"  Standard Accuracy:  {ibp_std_acc:.2f}%")
print(f"  Robust Accuracy:    {ibp_rob_acc:.2f}% (PGD with ε=0.1)")
print(f"\nStandard Model:")
print(f"  Standard Accuracy:  {std_std_acc:.2f}%")
print(f"  Robust Accuracy:    {std_rob_acc:.2f}% (PGD with ε=0.1)")
print(f"\nImprovement:")
print(f"  Robust Accuracy Gain: {ibp_rob_acc - std_rob_acc:+.2f}%")
print(f"\nTraining Time:")
print(f"  IBP Training:      {ibp_training_time:.2f}s ({ibp_training_time/60:.2f} min)")
print(f"  Standard Training: {standard_training_time:.2f}s ({standard_training_time/60:.2f} min)")
print(f"  Overhead:          {ibp_training_time/standard_training_time:.2f}x")
print("="*70)

PART (a): EVALUATION - Standard and Robust Accuracy

Evaluating IBP Model...


Testing: 100%|██████████| 157/157 [00:06<00:00, 24.59it/s]



Evaluating Standard Model...


Testing: 100%|██████████| 157/157 [00:06<00:00, 24.71it/s]


RESULTS SUMMARY - PART (a)

IBP Model:
  Standard Accuracy:  95.50%
  Robust Accuracy:    83.84% (PGD with ε=0.1)

Standard Model:
  Standard Accuracy:  97.26%
  Robust Accuracy:    0.90% (PGD with ε=0.1)

Improvement:
  Robust Accuracy Gain: +82.94%

Training Time:
  IBP Training:      7240.70s (120.68 min)
  Standard Training: 136.25s (2.27 min)
  Overhead:          53.14x


part b

In [17]:
def verify_robustness_single_image(model, x, y, epsilon):
    """
    Verify if a single image is certifiably robust using IBP
    Matches HW-2 box verification implementation
    """
    # Flatten input (no normalization - IBP model trained without it)
    x_flat = x.view(1, -1)
    
    # Create input bounds (same as HW-2)
    input_bounds = HyperRectangle.from_eps(x_flat, epsilon)
    
    # Create sequential model for bound propagation (same structure as HW-2)
    bound_model = nn.Sequential(
        model.fc1,
        nn.ReLU(),
        model.fc2,
        nn.ReLU(),
        model.fc3,
        nn.ReLU(),
        model.fc4
    )
    
    # Build bounded model (same as HW-2)
    factory = BoundModelFactory()
    bounded_net = factory.build(bound_model)
    
    # Perform IBP (same as HW-2)
    bounds = bounded_net.ibp(input_bounds)
    
    # Extract bounds
    lower = bounds.lower[0]
    upper = bounds.upper[0]
    
    # Verification logic (same as HW-2)
    label_lower = lower[y]
    
    for i in range(lower.shape[0]):
        if i != y and upper[i] >= label_lower:
            return False
    
    return True

def compute_verified_accuracy(model, test_loader, epsilon):
    """
    Compute verified accuracy (same logic as HW-2)
    """
    model.eval()
    num_verified = 0
    num_correct = 0
    
    for x, y in test_loader:
        x, y = x.to(device), y.to(device)
        
        for i in range(x.shape[0]):
            img = x[i:i+1]
            label = y[i].item()
            
            # Only verify correctly classified samples
            with torch.no_grad():
                pred = model(img).argmax(dim=1).item()
            
            if pred == label:
                num_correct += 1
                if verify_robustness_single_image(model, img, label, epsilon):
                    num_verified += 1
    
    verified_acc = 100.0 * num_verified / num_correct if num_correct > 0 else 0.0
    return num_verified, num_correct, verified_acc

In [18]:
# Test verified accuracy for 10 epsilon values from 0.01 to 0.1
print("="*70)
print("PART (b): VERIFIED ACCURACY - Box Verification")
print("="*70)

epsilon_values = np.linspace(0.01, 0.1, 10)

results = []
for idx, eps in enumerate(epsilon_values, 1):
    print(f"\n[{idx}/10] Testing ε = {eps:.4f}...")
    num_verified, num_correct, verified_acc = compute_verified_accuracy(model, test_loader, eps)
    results.append((eps, num_verified, num_correct, verified_acc))
    print(f"  Verified: {num_verified}/{num_correct} ({verified_acc:.2f}%)")

# Display results table
print("\n" + "="*70)
print("VERIFIED ACCURACY RESULTS")
print("="*70)
print(f"{'Epsilon':<15} {'Verified':<15} {'Correct':<15} {'Verified Acc':<15}")
print("-"*70)
for eps, num_ver, num_cor, ver_acc in results:
    print(f"{eps:<15.4f} {num_ver:<15d} {num_cor:<15d} {ver_acc:<15.2f}%")
print("="*70)

# Store results for later analysis
verified_results = results

PART (b): VERIFIED ACCURACY - Box Verification

[1/10] Testing ε = 0.0100...
  Verified: 9443/9550 (98.88%)

[2/10] Testing ε = 0.0200...
  Verified: 9260/9550 (96.96%)

[3/10] Testing ε = 0.0300...
  Verified: 9053/9550 (94.80%)

[4/10] Testing ε = 0.0400...
  Verified: 8783/9550 (91.97%)

[5/10] Testing ε = 0.0500...
  Verified: 8451/9550 (88.49%)

[6/10] Testing ε = 0.0600...
  Verified: 8064/9550 (84.44%)

[7/10] Testing ε = 0.0700...
  Verified: 7499/9550 (78.52%)

[8/10] Testing ε = 0.0800...
  Verified: 6731/9550 (70.48%)

[9/10] Testing ε = 0.0900...
  Verified: 5767/9550 (60.39%)

[10/10] Testing ε = 0.1000...
  Verified: 4574/9550 (47.90%)

VERIFIED ACCURACY RESULTS
Epsilon         Verified        Correct         Verified Acc   
----------------------------------------------------------------------
0.0100          9443            9550            98.88          %
0.0200          9260            9550            96.96          %
0.0300          9053            9550            94